In [ ]:
import os
from tqdm import tqdm
import torch
import torchvision
from torchsummary import summary
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import math
import json
import cv2
from torch.nn import functional as F

In [ ]:
IMG_SIZE = 256
IMG_DIR = "data/prep/S_224_R_2_C_False"
BATCH_SIZE = 64
NUM_WORKERS = os.cpu_count() - 2
USE_AMP = True  # Automatic Mixed Precision
EPOCHS = 1
BINARY = True

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Assuming that we are on a CUDA machine, this should print a CUDA device:

print(device)

In [ ]:
def get_folder_structure(base="data"):
    """
    Returns the folder structure of the data folder
    """
    return [x[0] for x in os.walk(base)]


get_folder_structure()

In [ ]:
class CImageFolderDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, classes, transform=None):
        self.root_dir = root_dir
        self.classes = classes
        self.transform = transform
        self.image_paths = []
        self.labels = []

        for class_idx, class_name in enumerate(classes):
            class_dir = os.path.join(root_dir, class_name)
            if os.path.isdir(class_dir):
                for image_name in os.listdir(class_dir):
                    image_path = os.path.join(class_dir, image_name)
                    self.image_paths.append(image_path)
                    self.labels.append(
                        class_idx
                    )  # Assign the corresponding class index

        self.class_to_idx = {
            cls: i for i, cls in enumerate(classes)
        }  # mapping class name to index

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = Image.open(image_path).convert("RGB")
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
if BINARY:
    classes = ["0", "4"]
else:
    classes = ["0", "1", "2", "3", "4"]
# Load the training data from the data folder
trainset = CImageFolderDataset(
    root_dir=os.path.join(IMG_DIR, "train/"),
    classes=classes,
    transform=torchvision.transforms.ToTensor(),
)
trainloader = torch.utils.data.DataLoader(
    trainset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
testset = CImageFolderDataset(
    root_dir=os.path.join(IMG_DIR, "test/"),
    classes=classes,
    transform=torchvision.transforms.ToTensor(),
)
testloader = torch.utils.data.DataLoader(
    testset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

In [ ]:
print("Following classes are there : \n", trainset.classes)

# Calculate the number of samples in each class
class_counts = {}
for label in trainset.labels:
    if trainset.classes[label] in class_counts:
        class_counts[trainset.classes[label]] += 1
    else:
        class_counts[trainset.classes[label]] = 1

# Print class distribution
print("\nClass distribution in training dataset:")
for class_name, count in class_counts.items():
    print(f"Class {class_name}: {count} samples")

# Visualize class distribution
plt.figure(figsize=(10, 6))
plt.bar(class_counts.keys(), class_counts.values())
plt.title("Class Distribution in Training Dataset")
plt.xlabel("Class")
plt.ylabel("Number of Samples")
plt.grid(axis="y", alpha=0.3)
for i, v in enumerate(class_counts.values()):
    plt.text(i, v + 5, str(v), ha="center")
plt.show()

# Calculate the same for test dataset
test_class_counts = {}
for label in testset.labels:
    if testset.classes[label] in test_class_counts:
        test_class_counts[testset.classes[label]] += 1
    else:
        test_class_counts[testset.classes[label]] = 1

print("\nClass distribution in test dataset:")
for class_name, count in test_class_counts.items():
    print(f"Class {class_name}: {count} samples")

In [ ]:
def display_img(img, label):
    print(f"Label : {trainset.classes[label]}")
    img_valid_range = np.clip(img.numpy().transpose((1, 2, 0)), 0, 1)
    plt.imshow(img_valid_range)


# display the first image in the dataset
display_img(*trainset[0])

In [ ]:
num_classes = len(trainset.classes)
# Cargar un modelo preentrenado
model = torchvision.models.resnet50(
    weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2
)

# Freeze all layers except the last ones
# for param in model.parameters():
#     param.requires_grad = False

# Modify the final layer for our classification task
model.fc = torch.nn.Linear(model.fc.in_features, num_classes)

# Move model to device
model = model.to(device)
# model = torch.compile(model)

In [ ]:
summary(model, (3, IMG_SIZE, IMG_SIZE), device=device.type)

In [ ]:
# criterion = torch.nn.NLLLoss(reduction="sum")  #  Use NLLLoss for binary-class classification
criterion = (
    torch.nn.CrossEntropyLoss()
)  # Use CrossEntropy for multi-class classification
optimizer = torch.optim.Adam(
    model.fc.parameters(), lr=1e-4, weight_decay=1e-5
)  # Only optimize parameters of final layer
lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=5
)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

In [ ]:
torch.set_num_threads(NUM_WORKERS)  # Usa todos los cores
torch.backends.cudnn.benchmark = True  # Mejora la velocidad de entrenamiento en GPU

In [ ]:
def train_model(model, criterion, optimizer, num_epochs=10):
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in tqdm(trainloader):
            inputs, labels = inputs.to(device), labels.to(device)
            for param in model.parameters():
                param.grad = None
            with torch.autocast(
                device_type="cuda", dtype=torch.float16, enabled=USE_AMP
            ):
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            # loss.backward()
            # optimizer.step()
            # optimizer.zero_grad()
            running_loss += loss.item() * inputs.size(0)
        epoch_loss = running_loss / len(trainloader.dataset)
        print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {epoch_loss:.4f}")
        # Validation loop
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():  # No gradients needed for validation
            for inputs, labels in testloader:
                inputs, labels = (
                    inputs.to(device),
                    labels.to(device),
                )  # to_device moved outside autograd
                with torch.autocast(
                    device_type="cuda", dtype=torch.float16, enabled=USE_AMP
                ):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_loss = val_loss / len(testloader.dataset)
        accuracy = 100 * correct / total
        print(f"Validation Loss: {val_loss:.4f}, Accuracy: {accuracy:.2f}%")
        lr_scheduler.step(val_loss)  # Adjust learning rate based on validation loss
        current_lr = optimizer.param_groups[0]["lr"]
        print(f"Current Learning Rate: {current_lr:.6f}")

In [ ]:
train_model(model, criterion, optimizer, num_epochs=EPOCHS)

In [ ]:
# Save the trained model to disk
model_save_path = os.path.join("models", "resnet50_binary_classifier.pth")

# Create directory if it doesn't exist
os.makedirs("models", exist_ok=True)

# Save model state dictionary
torch.save(model.state_dict(), model_save_path)

# Optionally, save the entire model (architecture + weights)
torch.save(model, os.path.join("models", "resnet50_full_model.pt"))

In [ ]:
model.eval()  # Establece modo de evaluación
correct = 0
total = 0

with torch.no_grad():  # Desactiva el cálculo de gradientes para evaluación
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Precisión en el conjunto de prueba: {100 * correct / total:.2f}%")

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Evaluate the model and collect predictions
model.eval()
all_predictions = []
all_labels = []

with torch.no_grad():
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        all_predictions.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Create confusion matrix
cm = confusion_matrix(all_labels, all_predictions)
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=trainset.classes,
    yticklabels=trainset.classes,
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

# Print classification report
print("Classification Report:")
print(classification_report(all_labels, all_predictions, target_names=trainset.classes))

# Plot some misclassified examples
misclassified_indices = []
model.eval()

with torch.no_grad():
    for i, (images, labels) in enumerate(testloader):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        # Find indices of misclassified images in this batch
        incorrect_indices = torch.nonzero(predicted != labels).squeeze().cpu().numpy()

        if len(incorrect_indices.shape) == 0 and incorrect_indices.size > 0:
            # Handle case where there's only one misclassified example
            incorrect_indices = [incorrect_indices.item()]

        for idx in incorrect_indices:
            batch_idx = i * testloader.batch_size + idx
            if batch_idx < len(testset):
                misclassified_indices.append(batch_idx)

                # Only collect a few examples
                if len(misclassified_indices) >= 5:
                    break

        if len(misclassified_indices) >= 5:
            break

# Display misclassified examples
if misclassified_indices:
    plt.figure(figsize=(15, 10))
    for i, idx in enumerate(misclassified_indices[:5]):
        img, label = testset[idx]

        plt.subplot(1, 5, i + 1)
        img_numpy = img.numpy().transpose((1, 2, 0))
        plt.imshow(np.clip(img_numpy, 0, 1))

        with torch.no_grad():
            output = model(img.unsqueeze(0).to(device))
            _, predicted = torch.max(output, 1)

        plt.title(
            f"True: {trainset.classes[label]}\nPred: {trainset.classes[predicted.item()]}"
        )
        plt.axis("off")

    plt.suptitle("Misclassified Examples")
    plt.tight_layout()
    plt.show()

In [ ]:
# img_path = "data/prep/S_224_R_2_C_False/test/0/image_16.png"
img_path = "data/prep/S_224_R_2_C_False/test/4/image_1074.png"
image = cv2.imread(img_path)
orig_image = image.copy()

image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
height, width, _ = image.shape

In [ ]:
model.eval()
# define a function to store the feature maps at the final convolutional layer
activation = {}


def getActivation(name):
    # the hook signature
    def hook(model, input, output):
        activation[name] = output.detach()

    return hook


model.layer4.register_forward_hook(getActivation("final_conv"))

In [ ]:
transform = torchvision.transforms.ToTensor()
image_tensor = transform(image)
# add batch dimension
image_tensor = image_tensor.unsqueeze(0).to(device)
# forward pass through model
outputs = model(image_tensor)
# get the idx of predicted class
class_idx = F.softmax(outputs).argmax().item()
print(class_idx)  # output: 201
class_idx = 1

In [ ]:
# Fetch feature maps at the final convolutional layer
conv_features = activation["final_conv"]

# Fetch the learned weights at the final feed-forward layer
weight_fc = model.fc.weight.detach().cpu().numpy()

In [ ]:
def calculate_cam(feature_conv, weight_fc, class_idx):
    # generate the class activation maps upsample to 224x224
    size_upsample = (224, 224)

    bz, nc, h, w = feature_conv.shape

    # Get the weights for the target class and compute the dot product with flattened features
    cam = weight_fc[class_idx].dot(feature_conv.reshape((nc, h * w)).cpu().numpy())
    cam = cam.reshape(h, w)
    cam = cam - np.min(cam)
    cam_img = cam / np.max(cam)
    cam_img = np.uint8(255 * cam_img)
    output_cam = cv2.resize(cam_img, size_upsample)

    return output_cam


# generate class activation mapping
class_activation_map = calculate_cam(conv_features, weight_fc, class_idx)

In [ ]:
from ipywidgets import interact, FloatSlider


def visualize_cam(class_activation_map, width, height, orig_image):
    import matplotlib.pyplot as plt

    # Create the heatmap using the class activation map
    heatmap = cv2.applyColorMap(
        cv2.resize(class_activation_map, (width, height)), cv2.COLORMAP_JET
    )

    # Convert original image from BGR to RGB for display
    orig_image_rgb = cv2.cvtColor(orig_image, cv2.COLOR_BGR2RGB)

    # Interactive visualization function
    def show_blend(alpha=0.5):
        # Blend images based on alpha value from slider
        blended = cv2.addWeighted(orig_image, 1 - alpha, heatmap, alpha, 0)
        blended_rgb = cv2.cvtColor(blended.astype(np.uint8), cv2.COLOR_BGR2RGB)

        # Display both images side by side
        plt.figure(figsize=(12, 6))
        plt.subplot(1, 2, 1)
        plt.imshow(orig_image_rgb)
        plt.title("Original Image")
        plt.axis("off")

        plt.subplot(1, 2, 2)
        plt.imshow(blended_rgb)
        plt.title(f"Class Activation Map (Alpha: {alpha:.2f})")
        plt.axis("off")
        plt.tight_layout()
        plt.show()

    # Create the interactive slider widget
    interact(
        show_blend,
        alpha=FloatSlider(
            min=0,
            max=1.0,
            step=0.05,
            value=0.5,
            description="Blend:",
            style={"description_width": "initial"},
        ),
    )


# visualize result
visualize_cam(class_activation_map, width, height, orig_image)